In [ ]:
import os
import glob

# Method 1: Combine all .txt files in 'entertainment' folder
def combine_txt_files(folder_path='entertainment', output_file='combined_entertainment.txt'):
    """
    Combine all .txt files in the specified folder into a single file.
    
    Args:
        folder_path: Path to the folder containing .txt files (default: 'entertainment')
        output_file: Name of the output combined file (default: 'combined_entertainment.txt')
    """
    
    # Pattern to find all .txt files in the folder
    txt_files = glob.glob(os.path.join(folder_path, '*.txt'))
    
    if not txt_files:
        print(f"No .txt files found in '{folder_path}' folder.")
        return
    
    # Open output file in write mode
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for i, txt_file in enumerate(sorted(txt_files)):  # Sorted for consistent order
            try:
                # Add separator between files (optional)
                if i > 0:
                    outfile.write(f"\n{'='*50}\n")  # Separator line
                    outfile.write(f"--- Start of {os.path.basename(txt_file)} ---\n")
                
                # Read content from each .txt file and write to output
                with open(txt_file, 'r', encoding='utf-8') as infile:
                    content = infile.read()
                    outfile.write(content)
                    
                print(f"Added: {txt_file}")
                
            except Exception as e:
                print(f"Error reading {txt_file}: {e}")
    
    print(f"\n✓ Successfully combined {len(txt_files)} files into '{output_file}'")

# Method 2: Without separators (just raw concatenation)
def combine_txt_files_simple(folder_path='lifestyle', output_file='combined_lifestyle.txt'):
    """Combine all .txt files without any separators."""
    
    txt_files = glob.glob(os.path.join(folder_path, '*.txt'))
    
    if not txt_files:
        print(f"No .txt files found in '{folder_path}' folder.")
        return
    
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for txt_file in sorted(txt_files):
            with open(txt_file, 'r', encoding='utf-8') as infile:
                outfile.write(infile.read())
    
    print(f"✓ Combined {len(txt_files)} files into '{output_file}'")

def main():
    """Run with command line arguments"""
    import sys
    
    folder = 'lifestyle'
    output =  'combined_lifestyle.txt'
    
    combine_txt_files_simple(folder, output)


if __name__ == "__main__":
    combine_txt_files_simple() 


✓ Combined 200 files into 'combined_lifestyle.txt'


In [3]:
def remove_metadata_lines():
    input_file = 'combined_entertainment.txt'
    output_file = 'cleaned_entertainment.txt'
    
    with open(input_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    # Lines to remove (containing these keywords)
    metadata_keywords = ['SITE', 'URL', 'CATEGORY', 'ARTICLE', 'TITLE', 'PARAGRAPHS', 'LENGTH']
    
    cleaned_lines = []
    for line in lines:
        # Keep the line if it doesn't contain any metadata keyword
        if not any(keyword in line for keyword in metadata_keywords):
            cleaned_lines.append(line)
    
    with open(output_file, 'w', encoding='utf-8') as file:
        file.writelines(cleaned_lines)
    
    print(f"✓ Removed metadata lines. Saved to '{output_file}'")

remove_metadata_lines()

✓ Removed metadata lines. Saved to 'cleaned_entertainment.txt'


In [4]:
def keep_only_article_content():
    input_file = 'cleaned_entertainment.txt'
    output_file = 'cleaned_entertainment_again.txt'
    
    with open(input_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    # Lines/patterns to remove
    patterns_to_remove = [
        '==================================================',
        '============================================================',
        '--- Start of article_'
    ]
    
    cleaned_lines = []
    for line in lines:
        # Check if line should be removed
        should_remove = False
        for pattern in patterns_to_remove:
            if pattern in line:
                should_remove = True
                break
        
        # Also remove empty lines (optional)
        if not should_remove and line.strip():  # Keep non-empty lines only
            cleaned_lines.append(line)
    
    with open(output_file, 'w', encoding='utf-8') as file:
        file.writelines(cleaned_lines)
    
    print(f"✓ Kept only article content. Saved to '{output_file}'")

keep_only_article_content()

✓ Kept only article content. Saved to 'cleaned_entertainment_again.txt'


In [2]:
import subprocess
from pathlib import Path
from tqdm import tqdm

INPUT_FILE = "cleaned_entertainment_again.txt"
OUTPUT_FILE = "translated_entertainment_english.txt"

MODEL = "llama3:latest"
BATCH_SIZE =  32

def call_ollama(prompt: str) -> str:
    """Call Ollama CLI with llama3 model"""
    result = subprocess.run(
        ["ollama", "run", MODEL],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return result.stdout.decode("utf-8").strip()

def build_prompt(batch):
    """Create a strict translation prompt"""
    numbered_lines = "\n".join(
        [f"{i+1}. {line}" for i, line in enumerate(batch)]
    )

    prompt = f"""
You are a professional translator.

Task:
Translate the following Nepali (Devanagari) text into precise, natural English.

Rules:
- Translate line-by-line
- Keep numbering EXACTLY the same
- Do NOT skip lines
- Do NOT add explanations
- Do NOT hallucinate
- Preserve meaning faithfully

Input:
{numbered_lines}

Output:
"""
    return prompt.strip()

def parse_output(output, batch_size):
    """Extract clean translated lines"""
    lines = output.split("\n")
    cleaned = []

    for line in lines:
        if "." in line:
            parts = line.split(".", 1)
            cleaned.append(parts[1].strip())

    # fallback if model messes formatting
    if len(cleaned) != batch_size:
        return [l.strip() for l in lines if l.strip()]

    return cleaned

def main():
    input_path = Path(INPUT_FILE)
    lines = [l.strip() for l in input_path.read_text(encoding="utf-8").splitlines() if l.strip()]

    print(f"Total lines: {len(lines)}")

    all_outputs = []

    for i in tqdm(range(0, len(lines), BATCH_SIZE)):
        batch = lines[i:i+BATCH_SIZE]

        prompt = build_prompt(batch)
        raw_output = call_ollama(prompt)

        translations = parse_output(raw_output, len(batch))
        all_outputs.extend(translations)

    # Save output
    Path(OUTPUT_FILE).write_text("\n".join(all_outputs), encoding="utf-8")

    print(f"Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Total lines: 3169


100%|██████████| 100/100 [6:56:02<00:00, 249.62s/it] 

Saved to translated_entertainment_english.txt


In [4]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
RAG_INSTRUCTION = "Answer ONLY using the provided context. If the answer is not found in the context, say 'I cannot find this information in the provided context.'"

def ask(context: str, question: str) -> str:
    prompt = f"""{RAG_INSTRUCTION}

Context: {context}
Question: {question}

Answer:"""

    response = requests.post(OLLAMA_URL, json={
        "model": "mistral-small3.1:latest",
        "prompt": prompt,
        "stream": False
    })
    
    return response.json()["response"].strip()


# ── Test Cases ───────────────────────────────────────────────────────

# Test 1: Answer IS in context (should answer)
print("=" * 50)
print("TEST 1: Answer in context")
print("=" * 50)
r1 = ask(
    context="New debit cards are dispatched within 3-5 working days after approval. You will receive a tracking SMS once dispatched.",
    question="I am still waiting on my card, when will it arrive?"
)
print(r1)


TEST 1: Answer in context
I cannot find this information in the provided context.


In [8]:
import requests
import pandas as pd
import json
import time

# ── Config ───────────────────────────────────────────────────────────
OLLAMA_URL      = "http://localhost:11434/api/generate"
MODEL           = "llama3"
RAG_INSTRUCTION = "Answer ONLY using the provided context. If the answer is not found in the context, say 'I cannot find this information in the provided context.'"

# ── Load Banking77 ───────────────────────────────────────────────────
train_df = pd.read_csv("https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv")
intent_queries = train_df.groupby("category")["text"].apply(list).to_dict()
intents        = list(intent_queries.keys())

# ── Take first 5 only ────────────────────────────────────────────────
test_intents  = intents[:5]
print("Testing with these 5 intents:")
for i, intent in enumerate(test_intents):
    print(f"  {i+1}. {intent} ({len(intent_queries[intent])} queries)")


# ── Ask Ollama ───────────────────────────────────────────────────────
def ask_ollama(prompt: str) -> str:
    response = requests.post(OLLAMA_URL, json={
        "model"  : MODEL,
        "prompt" : prompt,
        "stream" : False,
        "format" : "json"
    })
    return response.json()["response"].strip()


# ── Generate for one intent ──────────────────────────────────────────
def generate_intent_data(intent: str) -> dict:
    prompt = f"""
You are generating training data for a strict banking RAG chatbot.

Banking intent: "{intent}"

Generate a JSON object with these 3 fields:

1. "context": A realistic bank policy paragraph (2-3 sentences).

2. "answer": A clean chatbot response based ONLY on the context.

3. "unrelated_context": A different banking policy paragraph that does 
   NOT answer this intent.

STRICT RULES FOR "answer":
- Maximum 2-3 sentences
- Copy numbers and timeframes EXACTLY as they appear in context
  (if context says 7-10 days, answer must say 7-10 days, not "up to 10")
- Use ONLY information from your context
- Do NOT add anything not in the context
- Do NOT say "according to the context" or quote it
- Do NOT guess or estimate
- Be direct and professional
- No preamble, start with the answer directly

BAD answer: "Your refund may take up to 10 business days"  ← changed 7-10 to 10
GOOD answer: "Your refund will be processed within 7-10 business days."

Return ONLY valid JSON, nothing else:
{{
    "context": "...",
    "answer": "...",
    "unrelated_context": "..."
}}
"""
    try:
        raw    = ask_ollama(prompt)
        clean  = raw.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)

        assert "context"           in parsed
        assert "answer"            in parsed
        assert "unrelated_context" in parsed

        return parsed

    except Exception as e:
        print(f"  ⚠️  Failed: {e}")
        print(f"  Raw output was: {raw[:200]}")   # helps debug
        return None


# ── Run on 5 intents ─────────────────────────────────────────────────
print("\nGenerating...\n")

intent_data = {}

for i, intent in enumerate(test_intents):
    print(f"[{i+1}/5] {intent}")
    result = generate_intent_data(intent)

    if result:
        intent_data[intent] = result
        print(f"  context  : {result['context'][:80]}...")
        print(f"  answer   : {result['answer'][:80]}...")
        print(f"  unrelated: {result['unrelated_context'][:80]}...")
        print(f"  ✅ OK\n")
    else:
        print(f"  ❌ Failed\n")

    time.sleep(0.5)


# ── Build samples from 5 intents only ───────────────────────────────
samples = []

for intent in test_intents:
    if intent not in intent_data:
        continue

    data    = intent_data[intent]
    queries = intent_queries[intent]

    for query in queries:

        # Found in context
        samples.append({
            "instruction" : RAG_INSTRUCTION,
            "input"       : f"Context: {data['context']}\nQuestion: {query}",
            "output"      : data["answer"]
        })

        # NOT found in context
        samples.append({
            "instruction" : RAG_INSTRUCTION,
            "input"       : f"Context: {data['unrelated_context']}\nQuestion: {query}",
            "output"      : "I cannot find this information in the provided context."
        })


# ── Print results ────────────────────────────────────────────────────
print("=" * 60)
print(f"RESULTS")
print("=" * 60)
print(f"Intents generated : {len(intent_data)}/5")
print(f"Total samples     : {len(samples)}")
print()

print("── 4 Sample Previews ───────────────────────────────────────")
for i, s in enumerate(samples[:4]):
    print(f"\nSample {i+1}:")
    print(f"  instruction : {s['instruction'][:60]}...")
    print(f"  input       : {s['input'][:100]}...")
    print(f"  output      : {s['output']}")
    print()

Testing with these 5 intents:
  1. Refund_not_showing_up (162 queries)
  2. activate_my_card (159 queries)
  3. age_limit (110 queries)
  4. apple_pay_or_google_pay (126 queries)
  5. atm_support (87 queries)

Generating...

[1/5] Refund_not_showing_up
  context  : If you've requested a refund and it's not showing up in your account, please not...
  answer   : Your refund will be processed within 7-10 business days....
  unrelated: When using our mobile banking app, you can set up recurring transfers by selecti...
  ✅ OK

[2/5] activate_my_card
  context  : For security and verification purposes, we may require additional information or...
  answer   : Your activation will be processed within 7-10 business days....
  unrelated: We are committed to providing a secure online banking experience for our custome...
  ✅ OK

[3/5] age_limit
  context  : We understand that sometimes, you might need to request a refund. In this case, ...
  answer   : Your refund will be processed within 7-10 bu

In [9]:
intent_data

{'Refund_not_showing_up': {'context': "If you've requested a refund and it's not showing up in your account, please note that refunds are typically processed within 7-10 business days. During this time, we'll investigate the issue to ensure the refund is delivered as soon as possible.",
  'answer': 'Your refund will be processed within 7-10 business days.',
  'unrelated_context': "When using our mobile banking app, you can set up recurring transfers by selecting the 'Recurring' option from the transfer menu. This feature allows you to schedule regular payments to your own accounts or others."},
 'activate_my_card': {'context': 'For security and verification purposes, we may require additional information or documentation to activate your card. This process typically takes 7-10 business days from the date of request.',
  'answer': 'Your activation will be processed within 7-10 business days.',
  'unrelated_context': 'We are committed to providing a secure online banking experience for o

In [2]:
import sys
import re
import time
from pathlib import Path
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from enum import Enum

class RomanizationStyle(Enum):
    INFORMAL = "informal"     
    IAST = "iast"              # Academic with diacritics
    HARVARD_KYOTO = "harvard"  # ASCII with capitals
    ITRANS = "itrans"          # ITRANS standard
    NLRC = "nlrc"              # Nepali Language Resource Center

class DevanagariToRomanizedNepali:
    def __init__(self, model_name: str = "mistral-small3.1:latest", 
                 temperature: float = 0.0, 
                 batch_size: int = 10,
                 romanization_style: RomanizationStyle = RomanizationStyle.INFORMAL):
        self.model_name = model_name
        self.temperature = temperature
        self.batch_size = batch_size
        self.romanization_style = romanization_style
        self.llm = None
        self._initialize_model()

    def _initialize_model(self):
        try:
            self.llm = ChatOllama(
                model=self.model_name,
                temperature=self.temperature,
                num_predict=2048,
                num_ctx=4096,
            )
            print(f"✓ Model '{self.model_name}' loaded", file=sys.stderr)
            print(f"✓ Batch size: {self.batch_size} lines per API call", file=sys.stderr)
            print(f"✓ Romanization style: {self.romanization_style.value}", file=sys.stderr)
        except Exception as e:
            print(f"✗ Failed to load model: {e}", file=sys.stderr)
            sys.exit(1)

    def _get_style_instructions(self) -> str:
        """Get specific instructions based on romanization style"""
        
        styles = {
            RomanizationStyle.INFORMAL: """
ROMANIZATION STYLE: INFORMAL/INTERNET (most common for mobile/computer typing)
- Use 'ch' for च, 'chh' for छ
- Use 'b' for both ब and व (no distinction)
- Use 'gya' for ज्ञ
- Use 'tr' for त्र
- Use 'sh' for श and ष (no distinction)
- No diacritics (no ā, ī, ū, etc.)
- End words naturally without forcing 'a' sounds
- Examples:
  * 'के छ?' → 'ke chha?'
  * 'मलाई' → 'malai'
  * 'विद्यालय' → 'bidyalaya'
  * 'ज्ञान' → 'gyan'
  * 'कृपया' → 'kripaya'
  You are a native Nepali speaker from Kathmandu, Nepal with expert knowledge of Nepali phonetics and romanization systems.

YOUR TASK:
Convert Devanagari Nepali text to romanized Nepali.

YOUR IDENTITY:
- You speak NEPALI and have typed in romanized Nepali for 14+ years
- You understand all major romanization conventions
- You ALWAYS use Nepali words (not Hindi or Sanskrit)
""",
            
            RomanizationStyle.IAST: """
ROMANIZATION STYLE: IAST (International Alphabet of Sanskrit Transliteration) - Academic Standard
- Use diacritics: ā, ī, ū, ṛ, ṝ, ḷ, ḹ, ṃ, ḥ
- Distinguish: ś (श), ṣ (ष), s (स)
- Distinguish: ñ (ञ), ṇ (ण), n (न)
- Distinguish: ṭ (ट), ṭh (ठ), ḍ (ड), ḍh (ढ)
- Use 'c' for च, 'ch' for छ
- Use 'v' for व (distinguish from ब 'b')
- Examples:
  * 'के छ?' → 'ke cha?'
  * 'मलाई' → 'malāī'
  * 'विद्यालय' → 'vidyālaya'
  * 'ज्ञान' → 'jñāna'
  * 'कृपया' → 'kṛpayā'
""",
            
            RomanizationStyle.HARVARD_KYOTO: """
ROMANIZATION STYLE: Harvard-Kyoto (ASCII only, uses capitals for diacritics)
- Use capital letters for diacritics: A=ā, I=ī, U=ū, R=ṛ, RR=ṝ, L=ḷ, LL=ḹ
- Use 'z' for ś, 'S' for ṣ, 's' for s
- Use 'J' for ñ, 'N' for ṇ, 'n' for n
- Use 'T' for ṭ, 'Th' for ṭh, 'D' for ḍ, 'Dh' for ḍh
- Use 'c' for च, 'C' for छ
- Examples:
  * 'के छ?' → 'ke Ca?'
  * 'मलाई' → 'malAI'
  * 'विद्यालय' → 'vidyAlaya'
  * 'ज्ञान' → 'jJAna'
  * 'कृपया' → 'kRpayA'
""",
            
            RomanizationStyle.ITRANS: """
ROMANIZATION STYLE: ITRANS (popular for Devanagari to ASCII)
- Use 'aa' for ā, 'ii' for ī, 'uu' for ū, 'RRi' for ṛ
- Use 'sh' for श, 'Sh' for ष, 's' for स
- Use '~n' for ञ, 'N' for ण, 'n' for न
- Use 'T' for ट, 'Th' for ठ, 'D' for ड, 'Dh' for ढ
- Use 'ch' for च, 'Ch' for छ
- Use 'j~n' for ज्ञ
- Examples:
  * 'के छ?' → 'ke Ch?'
  * 'मलाई' → 'malaaii'
  * 'विद्यालय' → 'vidyaalaya'
  * 'ज्ञान' → 'j~naan'
  * 'कृपया' → 'kRipayaa'
""",
            
            RomanizationStyle.NLRC: """
ROMANIZATION STYLE: NLRC (Nepali Language Resource Center) - Nepali-specific
- Use apostrophe for vowel separation (e.g., 'ā'ī' for आई)
- Preserve schwa in certain contexts
- Use 'ba' for ब and 'wa' for व (distinguish)
- Use 'sha' for श, 'sha' for ष (no distinction)
- Use 'gya' for ज्ञ
- Examples:
  * 'के छ?' → 'ke cha?'
  * 'मलाई' → 'malā'ī'
  * 'विद्यालय' → 'widyālaya'
  * 'ज्ञान' → 'gyāna'
  * 'कृपया' → 'kṛpayā'
"""
        }
        
        return styles.get(self.romanization_style, styles[RomanizationStyle.INFORMAL])

    def _system_prompt(self):
        return f"""You are a native Nepali speaker from Kathmandu, Nepal with expert knowledge of Nepali phonetics and romanization systems.

YOUR TASK:
Convert Devanagari Nepali text to romanized Nepali.

YOUR IDENTITY:
- You speak NEPALI and have typed in romanized Nepali for 14+ years
- You understand all major romanization conventions
- You ALWAYS use Nepali words (not Hindi or Sanskrit)

{self._get_style_instructions()}

CRITICAL RULES:
1. The INPUT is in DEVANAGARI NEPALI script
2. Convert it to ROMANIZED NEPALI using the style specified above
3. Output ONLY the romanized text, no explanations, no extra text
4. Keep the exact same meaning and word order
5. Do NOT add any English words
6. Preserve numbers, punctuation, and spacing
7. Be consistent with the specified romanization rules
8. NEVER output Devanagari characters in the result"""

    def romanize_batch(self, texts: list) -> list:
        """Process multiple lines in one API call for better performance"""
        if not texts:
            return texts
        
        # Filter out empty lines but remember their positions
        non_empty_indices = [i for i, t in enumerate(texts) if t.strip()]
        empty_indices = [i for i, t in enumerate(texts) if not t.strip()]
        
        if not non_empty_indices:
            return texts
        
        # Prepare batch of non-empty texts
        batch_texts = [texts[i] for i in non_empty_indices]
        
        # Join with a unique separator
        separator = "\n###\n"
        batch_input = separator.join(batch_texts)
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", self._system_prompt() + f"\n\nConvert each of the following {len(batch_texts)} Devanagari Nepali lines to romanized Nepali using the specified style. Separate each output with '{separator.strip()}' on a new line. Keep the exact same order. No extra text or numbering."),
            ("human", "Convert these Devanagari Nepali lines to romanized Nepali:\n\n{text}")
        ])
        
        try:
            chain = prompt | self.llm | StrOutputParser()
            result = chain.invoke({"text": batch_input}).strip()
            
            # Split back into individual results
            batch_results = [r.strip() for r in result.split(separator)]
            
            if len(batch_results) != len(batch_texts):
                print(f"  ⚠ Batch size mismatch: expected {len(batch_texts)}, got {len(batch_results)}. Falling back to individual processing.", file=sys.stderr)
                batch_results = [self.romanize_single(text) for text in batch_texts]
            
            # Clean each result - remove any remaining Devanagari
            batch_results = [re.sub(r'[\u0900-\u097F]+', '', r).strip() or t for r, t in zip(batch_results, batch_texts)]
            
            # Reconstruct full results with empty lines in original positions
            full_results = [""] * len(texts)
            for idx, result_text in zip(non_empty_indices, batch_results):
                full_results[idx] = result_text
            for idx in empty_indices:
                full_results[idx] = texts[idx]  # Preserve empty lines
            
            return full_results
            
        except Exception as e:
            print(f"  ✗ Batch error: {e}. Falling back to individual processing.", file=sys.stderr)
            return [self.romanize_single(text) for text in texts]

    def romanize_single(self, text: str) -> str:
        """Fallback method for processing one line at a time"""
        if not text.strip():
            return text
        try:
            prompt = ChatPromptTemplate.from_messages([
                ("system", self._system_prompt() + "\n\nConvert this single Devanagari Nepali line to romanized Nepali using the specified style. Output ONLY the romanized text, no explanation."),
                ("human", "Devanagari: {text}\nRomanized:")
            ])
            chain = prompt | self.llm | StrOutputParser()
            result = chain.invoke({"text": text}).strip()
            # Strip any remaining Devanagari
            result = re.sub(r'[\u0900-\u097F]+', '', result).strip()
            return result if result else text
        except Exception as e:
            print(f"  ✗ Error on line: {e}", file=sys.stderr)
            return text

    def process_file(self, input_path: Path, output_path: Path, resume: bool = True):
        start_time = time.time()
        print(f"\n📄 Processing: {input_path.name}", file=sys.stderr)

        # Read input file (Devanagari Nepali)
        with open(input_path, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]

        total = len(lines)
        print(f"   Total lines: {total:,}", file=sys.stderr)

        processed_count = 0
        results = []

        if resume and output_path.exists():
            with open(output_path, 'r', encoding='utf-8') as f:
                results = [line.rstrip('\n') for line in f]
                processed_count = len(results)
                if processed_count < total:
                    print(f"   Resuming from line {processed_count:,} ({processed_count/total*100:.1f}% complete)", file=sys.stderr)
                    lines = lines[processed_count:]
                    results = results[:processed_count]
                else:
                    print(f"   ✓ Already complete, skipping.", file=sys.stderr)
                    return

        if not lines:
            return

        total_batches = (len(lines) + self.batch_size - 1) // self.batch_size
        print(f"   Processing {len(lines):,} lines in {total_batches} batches (batch size: {self.batch_size})", file=sys.stderr)
        
        with tqdm(total=len(lines), desc=f"  {input_path.name[:30]}", unit="lines", bar_format='{l_bar}{bar:35}{r_bar}') as pbar:
            for batch_start in range(0, len(lines), self.batch_size):
                batch_end = min(batch_start + self.batch_size, len(lines))
                batch_lines = lines[batch_start:batch_end]
                
                # Process the batch
                batch_results = self.romanize_batch(batch_lines)
                
                # Add to results
                results.extend(batch_results)
                
                # Update progress
                pbar.update(len(batch_lines))
                
                # Save checkpoint periodically
                if len(results) % 50 < self.batch_size:
                    self._save_results(results, output_path)
        
        # Final save
        self._save_results(results, output_path)

        elapsed = time.time() - start_time
        speed = total / elapsed if elapsed > 0 else 0
        print(f"   ✅ Done in {self._format_time(elapsed)} ({speed:.1f} lines/sec) -> {output_path}", file=sys.stderr)

    def _save_results(self, results: list, output_path: Path):
        """Save results to file atomically"""
        output_path.parent.mkdir(parents=True, exist_ok=True)
        temp_path = output_path.with_suffix('.tmp')
        with open(temp_path, 'w', encoding='utf-8') as f:
            for line in results:
                f.write(line + '\n')
        temp_path.replace(output_path)

    def _format_time(self, seconds: float) -> str:
        if seconds < 60:      return f"{seconds:.0f}s"
        elif seconds < 3600:  return f"{seconds/60:.1f}m"
        elif seconds < 86400: return f"{seconds/3600:.1f}h"
        else:                 return f"{seconds/86400:.1f}d"


def main():
    # ── Configuration ────────────────────────────────────────────────────────
    SOURCE_FOLDER  = "lifestyle"           # Folder containing Devanagari Nepali .txt files
    OUTPUT_FOLDER  = "lifestyle-romanized" # Folder for romanized output
    MODEL_NAME     = "mistral-small3.1:latest"
    BATCH_SIZE     = 10                  # Lines per API call (adjust based on performance)
    TEMPERATURE    = 0.0                 # 0 = deterministic, higher = more creative
    FILE_EXTENSION = ".txt"              # File extension to process
    RESUME         = True                # Resume from last checkpoint
    
    # Choose your romanization style:
    # Options: INFORMAL, IAST, HARVARD_KYOTO, ITRANS, NLRC
    ROMANIZATION_STYLE = RomanizationStyle.INFORMAL  # <-- CHANGE THIS
    # ─────────────────────────────────────────────────────────────────────────

    print("=" * 70, file=sys.stderr)
    print("  DEVANAGARI NEPALI → ROMANIZED NEPALI — BATCH PROCESSOR", file=sys.stderr)
    print("=" * 70, file=sys.stderr)

    source_path = Path(SOURCE_FOLDER)
    output_path = Path(OUTPUT_FOLDER)

    if not source_path.exists():
        print(f"✗ Error: Source folder '{SOURCE_FOLDER}' not found.", file=sys.stderr)
        sys.exit(1)

    input_files = sorted(source_path.glob(f"*{FILE_EXTENSION}"))
    if not input_files:
        print(f"✗ No {FILE_EXTENSION} files found in '{SOURCE_FOLDER}'.", file=sys.stderr)
        sys.exit(1)

    print(f"\n✓ Source folder: {source_path.resolve()}", file=sys.stderr)
    print(f"✓ Output folder: {output_path.resolve()}", file=sys.stderr)
    print(f"✓ Files found: {len(input_files)}", file=sys.stderr)
    print(f"✓ Model: {MODEL_NAME}", file=sys.stderr)
    print(f"✓ Romanization style: {ROMANIZATION_STYLE.value.upper()}", file=sys.stderr)
    print(f"✓ Batch size: {BATCH_SIZE} lines per API call", file=sys.stderr)

    # Show files to process
    print(f"\nFiles to process:", file=sys.stderr)
    for f in input_files:
        size_kb = f.stat().st_size / 1024
        out_f = output_path / f"{f.stem}_{ROMANIZATION_STYLE.value}{f.suffix}"
        status = "✓ done" if (RESUME and out_f.exists()) else "-> pending"
        # Check if it's Devanagari (basic check)
        with open(f, 'r', encoding='utf-8') as test_f:
            sample = test_f.read(100)
            has_devanagari = bool(re.search(r'[\u0900-\u097F]', sample))
            script_type = "Devanagari" if has_devanagari else "Unknown"
        print(f"  {status}  {f.name} ({size_kb:.1f} KB) - {script_type}", file=sys.stderr)

    romanizer = DevanagariToRomanizedNepali(MODEL_NAME, TEMPERATURE, BATCH_SIZE, ROMANIZATION_STYLE)
    output_path.mkdir(parents=True, exist_ok=True)

    grand_start = time.time()
    completed = 0

    for idx, in_file in enumerate(input_files, 1):
        # Output file with style name to distinguish different romanizations
        out_file = output_path / f"{in_file.stem}_{ROMANIZATION_STYLE.value}{in_file.suffix}"

        # Skip completely processed files
        if RESUME and out_file.exists():
            with open(in_file, 'r', encoding='utf-8') as fi:
                in_count = sum(1 for _ in fi)
            with open(out_file, 'r', encoding='utf-8') as fo:
                out_count = sum(1 for _ in fo)
            if in_count == out_count:
                print(f"\n[{idx}/{len(input_files)}] ⏭  Skipping (already complete): {in_file.name}", file=sys.stderr)
                completed += 1
                continue

        print(f"\n[{idx}/{len(input_files)}] Processing: {in_file.name}", file=sys.stderr)
        romanizer.process_file(in_file, out_file, resume=RESUME)
        completed += 1

    grand_elapsed = time.time() - grand_start
    print(f"\n{'=' * 70}", file=sys.stderr)
    print(f"  ALL DONE!", file=sys.stderr)
    print(f"  Files processed: {completed}/{len(input_files)}", file=sys.stderr)
    print(f"  Total time: {romanizer._format_time(grand_elapsed)}", file=sys.stderr)
    print(f"  Output folder: {output_path.resolve()}", file=sys.stderr)
    print(f"  Romanization style: {ROMANIZATION_STYLE.value}", file=sys.stderr)
    print(f"{'=' * 70}", file=sys.stderr)


if __name__ == "__main__":
    main()

  DEVANAGARI NEPALI → ROMANIZED NEPALI — BATCH PROCESSOR

✓ Source folder: /home/lang-chain/Documents/Astra_agentic_RAG/dataset_ne/lifestyle
✓ Output folder: /home/lang-chain/Documents/Astra_agentic_RAG/dataset_ne/lifestyle-romanized
✓ Files found: 200
✓ Model: mistral-small3.1:latest
✓ Romanization style: INFORMAL
✓ Batch size: 10 lines per API call

Files to process:
  -> pending  article_0001.txt (9.2 KB) - Unknown
  -> pending  article_0002.txt (11.0 KB) - Unknown
  -> pending  article_0003.txt (4.9 KB) - Unknown
  -> pending  article_0004.txt (11.2 KB) - Unknown
  -> pending  article_0005.txt (9.7 KB) - Unknown
  -> pending  article_0006.txt (4.0 KB) - Unknown
  -> pending  article_0007.txt (8.8 KB) - Unknown
  -> pending  article_0008.txt (17.2 KB) - Unknown
  -> pending  article_0009.txt (3.5 KB) - Unknown
  -> pending  article_0010.txt (2.4 KB) - Unknown
  -> pending  article_0011.txt (6.8 KB) - Unknown
  -> pending  article_0012.txt (4.3 KB) - Unknown
  -> pending  article_001

KeyboardInterrupt: 

In [ ]:
import sys
import re
import time
from pathlib import Path
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from enum import Enum

class RomanizationStyle(Enum):
    INFORMAL = "informal"     
    IAST = "iast"              # Academic with diacritics
    HARVARD_KYOTO = "harvard"  # ASCII with capitals
    ITRANS = "itrans"          # ITRANS standard
    NLRC = "nlrc"              # Nepali Language Resource Center

class DevanagariToRomanizedNepali:
    def __init__(self, model_name: str = "mistral-small3.1:latest", 
                 temperature: float = 0.0, 
                 batch_size: int = 10,
                 romanization_style: RomanizationStyle = RomanizationStyle.INFORMAL):
        self.model_name = model_name
        self.temperature = temperature
        self.batch_size = batch_size
        self.romanization_style = romanization_style
        self.llm = None
        self._initialize_model()

    def _initialize_model(self):
        try:
            self.llm = ChatOllama(
                model=self.model_name,
                temperature=self.temperature,
                num_predict=2048,
                num_ctx=4096,
            )
            print(f"✓ Model '{self.model_name}' loaded", file=sys.stderr)
            print(f"✓ Batch size: {self.batch_size} lines per API call", file=sys.stderr)
            print(f"✓ Romanization style: {self.romanization_style.value}", file=sys.stderr)
        except Exception as e:
            print(f"✗ Failed to load model: {e}", file=sys.stderr)
            sys.exit(1)

    # ------------------------------------------------------------------
    # Shared rules that apply regardless of romanization style. These are
    # the two biggest sources of "Hindi-sounding" / inconsistently-cased
    # output: schwa deletion (a Hindi phonology habit) and English-style
    # noun capitalization (which Devanagari has no concept of at all).
    # ------------------------------------------------------------------
    def _shared_rules(self) -> str:
        return """
CAPITALIZATION (Devanagari has no concept of case - do not invent one):
- Default to ALL LOWERCASE.
- Capitalize ONLY:
  1. The first word of a sentence.
  2. True proper nouns: personal names, place names, deity names,
     organization/brand names, days, months.
- Do NOT capitalize ordinary nouns, verbs, or adjectives just because
  they seem important or formal. This is a common mistake - avoid it.
- Examples:
  * "घर सफा छ।" -> "ghar safa cha." (house = common noun, lowercase)
  * "राम काठमाडौं गयो।" -> "Ram Kathmandu gayo." (Ram, Kathmandu = proper nouns, capitalized)
  * "यो राम्रो किताब हो।" -> "yo ramro kitab ho." (NOT "Yo Ramro Kitab Ho.")

SCHWA RETENTION (this is the single biggest thing that makes output
sound Nepali instead of Hindi - follow it carefully):
- Hindi aggressively DROPS the inherent schwa ("a") sound at the end of
  words and inside consonant clusters (काम -> "kaam", राम -> "raam").
- Nepali generally RETAINS that inherent "a" far more than Hindi does.
- When a word could be pronounced either the Hindi way or the Nepali
  way, always choose the NEPALI pronunciation, not the Hindi one.
- Contrastive examples (do NOT produce the Hindi-style form on the left):
  * ✗ Hindi-style: "vidyalay"   -> ✓ Nepali: "bidyalaya"
  * ✗ Hindi-style: "kaam"       -> ✓ Nepali: "kaam" is fine here, but
    ✗ Hindi-style: "ghar pe"    -> ✓ Nepali: "gharma" (Nepali case marker, not Hindi postposition)
  * ✗ Hindi-style: "keh raha hai" -> ✓ Nepali: "bhandai cha"
- Never substitute Hindi postpositions, verb forms, or vocabulary
  (e.g. "hai", "raha", "pe", "wala") even if the Devanagari spelling
  looks similar to a Hindi word. Always use the Nepali equivalent.
"""

    def _get_style_instructions(self) -> str:
        """Get specific instructions based on romanization style"""
        
        styles = {
            RomanizationStyle.INFORMAL: """
ROMANIZATION STYLE: INFORMAL/INTERNET (most common for mobile/computer typing)
- Use 'ch' for च, 'chh' for छ
- Use 'b' for both ब and व (no distinction)
- Use 'gya' for ज्ञ
- Use 'tr' for त्र
- Use 'sh' for श and ष (no distinction)
- No diacritics (no ā, ī, ū, etc.)
- End words naturally without forcing 'a' sounds
- Examples:
  * 'के छ?' → 'ke chha?'
  * 'मलाई' → 'malai'
  * 'विद्यालय' → 'bidyalaya'
  * 'ज्ञान' → 'gyan'
  * 'कृपया' → 'kripaya'
""",
            
            RomanizationStyle.IAST: """
ROMANIZATION STYLE: IAST (International Alphabet of Sanskrit Transliteration) - Academic Standard
- Use diacritics: ā, ī, ū, ṛ, ṝ, ḷ, ḹ, ṃ, ḥ
- Distinguish: ś (श), ṣ (ष), s (स)
- Distinguish: ñ (ञ), ṇ (ण), n (न)
- Distinguish: ṭ (ट), ṭh (ठ), ḍ (ड), ḍh (ढ)
- Use 'c' for च, 'ch' for छ
- Use 'v' for व (distinguish from ब 'b')
- Examples:
  * 'के छ?' → 'ke cha?'
  * 'मलाई' → 'malāī'
  * 'विद्यालय' → 'vidyālaya'
  * 'ज्ञान' → 'jñāna'
  * 'कृपया' → 'kṛpayā'
""",
            
            RomanizationStyle.HARVARD_KYOTO: """
ROMANIZATION STYLE: Harvard-Kyoto (ASCII only, uses capitals for diacritics ONLY - see note below)
- Use capital letters ONLY for diacritic substitution, not for nouns:
  A=ā, I=ī, U=ū, R=ṛ, RR=ṝ, L=ḷ, LL=ḹ
- Use 'z' for ś, 'S' for ṣ, 's' for s
- Use 'J' for ñ, 'N' for ṇ, 'n' for n
- Use 'T' for ṭ, 'Th' for ṭh, 'D' for ḍ, 'Dh' for ḍh
- Use 'c' for च, 'C' for छ
- NOTE: capital letters here are phonetic substitutions ONLY (e.g. 'A' means ā,
  not "this is a proper noun"). Do NOT additionally apply English-style
  capitalization rules on top of this - the CAPITALIZATION section above
  about proper nouns still applies for word-initial capitals.
- Examples:
  * 'के छ?' → 'ke Ca?'
  * 'मलाई' → 'malAI'
  * 'विद्यालय' → 'vidyAlaya'
  * 'ज्ञान' → 'jJAna'
  * 'कृपया' → 'kRpayA'
""",
            
            RomanizationStyle.ITRANS: """
ROMANIZATION STYLE: ITRANS (popular for Devanagari to ASCII)
- Use 'aa' for ā, 'ii' for ī, 'uu' for ū, 'RRi' for ṛ
- Use 'sh' for श, 'Sh' for ष, 's' for स
- Use '~n' for ञ, 'N' for ण, 'n' for न
- Use 'T' for ट, 'Th' for ठ, 'D' for ड, 'Dh' for ढ
- Use 'ch' for च, 'Ch' for छ
- Use 'j~n' for ज्ञ
- Examples:
  * 'के छ?' → 'ke Ch?'
  * 'मलाई' → 'malaaii'
  * 'विद्यालय' → 'vidyaalaya'
  * 'ज्ञान' → 'j~naan'
  * 'कृपया' → 'kRipayaa'
""",
            
            RomanizationStyle.NLRC: """
ROMANIZATION STYLE: NLRC (Nepali Language Resource Center) - Nepali-specific
- Use apostrophe for vowel separation (e.g., 'ā'ī' for आई)
- Preserve schwa in certain contexts
- Use 'ba' for ब and 'wa' for व (distinguish)
- Use 'sha' for श, 'sha' for ष (no distinction)
- Use 'gya' for ज्ञ
- Examples:
  * 'के छ?' → 'ke cha?'
  * 'मलाई' → 'malā'ī'
  * 'विद्यालय' → 'widyālaya'
  * 'ज्ञान' → 'gyāna'
  * 'कृपया' → 'kṛpayā'
"""
        }
        
        return styles.get(self.romanization_style, styles[RomanizationStyle.INFORMAL])

    def _system_prompt(self):
        return f"""You are a native Nepali speaker from Kathmandu, Nepal with expert knowledge of Nepali phonetics and romanization systems.

YOUR TASK:
Convert Devanagari Nepali text to romanized Nepali.

YOUR IDENTITY:
- You speak NEPALI and have typed in romanized Nepali for 14+ years
- You understand all major romanization conventions
- You ALWAYS use Nepali words, grammar, and pronunciation - NEVER Hindi or Sanskrit

{self._shared_rules()}

{self._get_style_instructions()}

CRITICAL RULES:
1. The INPUT is in DEVANAGARI NEPALI script
2. Convert it to ROMANIZED NEPALI using the style specified above
3. Output ONLY the romanized text, no explanations, no extra text
4. Keep the exact same meaning and word order
5. Do NOT add any English words
6. Preserve numbers, punctuation, and spacing
7. Be consistent with the specified romanization rules
8. NEVER output Devanagari characters in the result
9. Follow the CAPITALIZATION rules above exactly - do not capitalize
   common nouns/verbs/adjectives, only proper nouns and sentence-starts
10. Follow the SCHWA RETENTION rules above - if a word could sound
    Hindi or Nepali, always pick the Nepali pronunciation/spelling"""

    def romanize_batch(self, texts: list) -> list:
        """Process multiple lines in one API call for better performance"""
        if not texts:
            return texts
        
        # Filter out empty lines but remember their positions
        non_empty_indices = [i for i, t in enumerate(texts) if t.strip()]
        empty_indices = [i for i, t in enumerate(texts) if not t.strip()]
        
        if not non_empty_indices:
            return texts
        
        # Prepare batch of non-empty texts
        batch_texts = [texts[i] for i in non_empty_indices]
        
        # Join with a unique separator
        separator = "\n###\n"
        batch_input = separator.join(batch_texts)
        
        # Reminder appended right next to the actual text (recency helps
        # keep long batches from drifting off the capitalization/schwa rules).
        reminder = (
            "\n\nReminder before you output: lowercase for all common nouns "
            "(capitalize ONLY proper nouns / sentence starts), and always use "
            "Nepali pronunciation/vocabulary, never Hindi."
        )

        prompt = ChatPromptTemplate.from_messages([
            ("system", self._system_prompt() + f"\n\nConvert each of the following {len(batch_texts)} Devanagari Nepali lines to romanized Nepali using the specified style. Separate each output with '{separator.strip()}' on a new line. Keep the exact same order. No extra text or numbering."),
            ("human", "Convert these Devanagari Nepali lines to romanized Nepali:\n\n{text}" + reminder)
        ])
        
        try:
            chain = prompt | self.llm | StrOutputParser()
            result = chain.invoke({"text": batch_input}).strip()
            
            # Split back into individual results
            batch_results = [r.strip() for r in result.split(separator)]
            
            if len(batch_results) != len(batch_texts):
                print(f"  ⚠ Batch size mismatch: expected {len(batch_texts)}, got {len(batch_results)}. Falling back to individual processing.", file=sys.stderr)
                batch_results = [self.romanize_single(text) for text in batch_texts]
            
            # Clean each result - remove any remaining Devanagari
            batch_results = [re.sub(r'[\u0900-\u097F]+', '', r).strip() or t for r, t in zip(batch_results, batch_texts)]
            
            # Reconstruct full results with empty lines in original positions
            full_results = [""] * len(texts)
            for idx, result_text in zip(non_empty_indices, batch_results):
                full_results[idx] = result_text
            for idx in empty_indices:
                full_results[idx] = texts[idx]  # Preserve empty lines
            
            return full_results
            
        except Exception as e:
            print(f"  ✗ Batch error: {e}. Falling back to individual processing.", file=sys.stderr)
            return [self.romanize_single(text) for text in texts]

    def romanize_single(self, text: str) -> str:
        """Fallback method for processing one line at a time"""
        if not text.strip():
            return text
        try:
            prompt = ChatPromptTemplate.from_messages([
                ("system", self._system_prompt() + "\n\nConvert this single Devanagari Nepali line to romanized Nepali using the specified style. Output ONLY the romanized text, no explanation."),
                ("human", "Devanagari: {text}\nRomanized:")
            ])
            chain = prompt | self.llm | StrOutputParser()
            result = chain.invoke({"text": text}).strip()
            # Strip any remaining Devanagari
            result = re.sub(r'[\u0900-\u097F]+', '', result).strip()
            return result if result else text
        except Exception as e:
            print(f"  ✗ Error on line: {e}", file=sys.stderr)
            return text

    def process_file(self, input_path: Path, output_path: Path, resume: bool = True):
        start_time = time.time()
        print(f"\n📄 Processing: {input_path.name}", file=sys.stderr)

        # Read input file (Devanagari Nepali)
        with open(input_path, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]

        total = len(lines)
        print(f"   Total lines: {total:,}", file=sys.stderr)

        processed_count = 0
        results = []

        if resume and output_path.exists():
            with open(output_path, 'r', encoding='utf-8') as f:
                results = [line.rstrip('\n') for line in f]
                processed_count = len(results)
                if processed_count < total:
                    print(f"   Resuming from line {processed_count:,} ({processed_count/total*100:.1f}% complete)", file=sys.stderr)
                    lines = lines[processed_count:]
                    results = results[:processed_count]
                else:
                    print(f"   ✓ Already complete, skipping.", file=sys.stderr)
                    return

        if not lines:
            return

        total_batches = (len(lines) + self.batch_size - 1) // self.batch_size
        print(f"   Processing {len(lines):,} lines in {total_batches} batches (batch size: {self.batch_size})", file=sys.stderr)
        
        with tqdm(total=len(lines), desc=f"  {input_path.name[:30]}", unit="lines", bar_format='{l_bar}{bar:35}{r_bar}') as pbar:
            for batch_start in range(0, len(lines), self.batch_size):
                batch_end = min(batch_start + self.batch_size, len(lines))
                batch_lines = lines[batch_start:batch_end]
                
                # Process the batch
                batch_results = self.romanize_batch(batch_lines)
                
                # Add to results
                results.extend(batch_results)
                
                # Update progress
                pbar.update(len(batch_lines))
                
                # Save checkpoint periodically
                if len(results) % 50 < self.batch_size:
                    self._save_results(results, output_path)
        
        # Final save
        self._save_results(results, output_path)

        elapsed = time.time() - start_time
        speed = total / elapsed if elapsed > 0 else 0
        print(f"   ✅ Done in {self._format_time(elapsed)} ({speed:.1f} lines/sec) -> {output_path}", file=sys.stderr)

    def _save_results(self, results: list, output_path: Path):
        """Save results to file atomically"""
        output_path.parent.mkdir(parents=True, exist_ok=True)
        temp_path = output_path.with_suffix('.tmp')
        with open(temp_path, 'w', encoding='utf-8') as f:
            for line in results:
                f.write(line + '\n')
        temp_path.replace(output_path)

    def _format_time(self, seconds: float) -> str:
        if seconds < 60:      return f"{seconds:.0f}s"
        elif seconds < 3600:  return f"{seconds/60:.1f}m"
        elif seconds < 86400: return f"{seconds/3600:.1f}h"
        else:                 return f"{seconds/86400:.1f}d"


def main():
    # ── Configuration ────────────────────────────────────────────────────────
    SOURCE_FOLDER  = "lifestyle"           # Folder containing Devanagari Nepali .txt files
    OUTPUT_FOLDER  = "lifestyle-romanized-new" # Folder for romanized output
    MODEL_NAME     = "mistral-small3.1:latest"
    BATCH_SIZE     = 10                  # Lines per API call (adjust based on performance)
    TEMPERATURE    = 0.0                 # 0 = deterministic, higher = more creative
    FILE_EXTENSION = ".txt"              # File extension to process
    RESUME         = True                # Resume from last checkpoint
    
    # Choose your romanization style:
    # Options: INFORMAL, IAST, HARVARD_KYOTO, ITRANS, NLRC
    ROMANIZATION_STYLE = RomanizationStyle.INFORMAL  # <-- CHANGE THIS
    # ─────────────────────────────────────────────────────────────────────────

    print("=" * 70, file=sys.stderr)
    print("  DEVANAGARI NEPALI → ROMANIZED NEPALI — BATCH PROCESSOR", file=sys.stderr)
    print("=" * 70, file=sys.stderr)

    source_path = Path(SOURCE_FOLDER)
    output_path = Path(OUTPUT_FOLDER)

    if not source_path.exists():
        print(f"✗ Error: Source folder '{SOURCE_FOLDER}' not found.", file=sys.stderr)
        sys.exit(1)

    input_files = sorted(source_path.glob(f"*{FILE_EXTENSION}"))
    if not input_files:
        print(f"✗ No {FILE_EXTENSION} files found in '{SOURCE_FOLDER}'.", file=sys.stderr)
        sys.exit(1)

    print(f"\n✓ Source folder: {source_path.resolve()}", file=sys.stderr)
    print(f"✓ Output folder: {output_path.resolve()}", file=sys.stderr)
    print(f"✓ Files found: {len(input_files)}", file=sys.stderr)
    print(f"✓ Model: {MODEL_NAME}", file=sys.stderr)
    print(f"✓ Romanization style: {ROMANIZATION_STYLE.value.upper()}", file=sys.stderr)
    print(f"✓ Batch size: {BATCH_SIZE} lines per API call", file=sys.stderr)

    # Show files to process
    print(f"\nFiles to process:", file=sys.stderr)
    for f in input_files:
        size_kb = f.stat().st_size / 1024
        out_f = output_path / f"{f.stem}_{ROMANIZATION_STYLE.value}{f.suffix}"
        status = "✓ done" if (RESUME and out_f.exists()) else "-> pending"
        # Check if it's Devanagari (basic check)
        with open(f, 'r', encoding='utf-8') as test_f:
            sample = test_f.read(100)
            has_devanagari = bool(re.search(r'[\u0900-\u097F]', sample))
            script_type = "Devanagari" if has_devanagari else "Unknown"
        print(f"  {status}  {f.name} ({size_kb:.1f} KB) - {script_type}", file=sys.stderr)

    romanizer = DevanagariToRomanizedNepali(MODEL_NAME, TEMPERATURE, BATCH_SIZE, ROMANIZATION_STYLE)
    output_path.mkdir(parents=True, exist_ok=True)

    grand_start = time.time()
    completed = 0

    for idx, in_file in enumerate(input_files, 1):
        # Output file with style name to distinguish different romanizations
        out_file = output_path / f"{in_file.stem}_{ROMANIZATION_STYLE.value}{in_file.suffix}"

        # Skip completely processed files
        if RESUME and out_file.exists():
            with open(in_file, 'r', encoding='utf-8') as fi:
                in_count = sum(1 for _ in fi)
            with open(out_file, 'r', encoding='utf-8') as fo:
                out_count = sum(1 for _ in fo)
            if in_count == out_count:
                print(f"\n[{idx}/{len(input_files)}] ⏭  Skipping (already complete): {in_file.name}", file=sys.stderr)
                completed += 1
                continue

        print(f"\n[{idx}/{len(input_files)}] Processing: {in_file.name}", file=sys.stderr)
        romanizer.process_file(in_file, out_file, resume=RESUME)
        completed += 1

    grand_elapsed = time.time() - grand_start
    print(f"\n{'=' * 70}", file=sys.stderr)
    print(f"  ALL DONE!", file=sys.stderr)
    print(f"  Files processed: {completed}/{len(input_files)}", file=sys.stderr)
    print(f"  Total time: {romanizer._format_time(grand_elapsed)}", file=sys.stderr)
    print(f"  Output folder: {output_path.resolve()}", file=sys.stderr)
    print(f"  Romanization style: {ROMANIZATION_STYLE.value}", file=sys.stderr)
    print(f"{'=' * 70}", file=sys.stderr)


if __name__ == "__main__":
    main()

  DEVANAGARI NEPALI → ROMANIZED NEPALI — BATCH PROCESSOR

✓ Source folder: /home/lang-chain/Documents/Astra_agentic_RAG/dataset_ne/lifestyle
✓ Output folder: /home/lang-chain/Documents/Astra_agentic_RAG/dataset_ne/lifestyle-romanized-new
✓ Files found: 200
✓ Model: llama3.2:3b
✓ Romanization style: INFORMAL
✓ Batch size: 10 lines per API call

Files to process:
  -> pending  article_0001.txt (9.2 KB) - Unknown
  -> pending  article_0002.txt (11.0 KB) - Unknown
  -> pending  article_0003.txt (4.9 KB) - Unknown
  -> pending  article_0004.txt (11.2 KB) - Unknown
  -> pending  article_0005.txt (9.7 KB) - Unknown
  -> pending  article_0006.txt (4.0 KB) - Unknown
  -> pending  article_0007.txt (8.8 KB) - Unknown
  -> pending  article_0008.txt (17.2 KB) - Unknown
  -> pending  article_0009.txt (3.5 KB) - Unknown
  -> pending  article_0010.txt (2.4 KB) - Unknown
  -> pending  article_0011.txt (6.8 KB) - Unknown
  -> pending  article_0012.txt (4.3 KB) - Unknown
  -> pending  article_0013.txt (3

In [3]:
from pathlib import Path

def concat_text_files_strip_header(folder_path="opinion", output_file="opinion.txt", extension=".txt",
                                   header_separator="============================================================",
                                   recursive=False):
    """
    Concatenate all text files with a given extension inside a folder,
    removing the metadata header (everything up to and including a separator line)
    from each file.

    Args:
        folder_path (str): Path to the folder containing the text files.
        output_file (str): Name of the output file (will be created/overwritten).
        extension (str): File extension to look for (e.g., ".txt", ".log").
        header_separator (str): The exact line that marks the end of the header.
        recursive (bool): If True, also search in subfolders.
    """
    folder = Path(folder_path)

    if not folder.exists() or not folder.is_dir():
        print(f"Error: '{folder_path}' is not a valid directory.")
        return

    # Collect files (sorted alphabetically)
    if recursive:
        files = sorted(folder.rglob(f"*{extension}"))
    else:
        files = sorted(folder.glob(f"*{extension}"))

    if not files:
        print(f"No files with extension '{extension}' found in '{folder_path}'.")
        return

    total_files = 0
    total_lines_written = 0

    with open(output_file, 'w', encoding='utf-8') as out_f:
        for file_path in files:
            try:
                with open(file_path, 'r', encoding='utf-8') as in_f:
                    lines = in_f.readlines()

                # Find the index of the header separator line
                separator_index = -1
                for i, line in enumerate(lines):
                    if header_separator in line:   # checks if the line contains the separator
                        separator_index = i
                        break

                # If separator found, take everything after it (skip that line too)
                if separator_index != -1:
                    content_lines = lines[separator_index + 1:]
                else:
                    # No separator found – maybe the header is not there; use the whole file
                    content_lines = lines

                # Write the content to the output file
                if content_lines:
                    out_f.writelines(content_lines)
                    # Add an extra newline between files for readability (optional)
                    if not content_lines[-1].endswith('\n'):
                        out_f.write('\n')
                    else:
                        out_f.write('\n')   # add a blank line between files
                else:
                    print(f"Warning: '{file_path}' has no content after the header.")

                total_files += 1
                total_lines_written += len(content_lines)

            except Exception as e:
                print(f"Could not process {file_path}: {e}")

    print(f"Successfully processed {total_files} files. Combined output written to '{output_file}'.")
    print(f"Total lines after stripping headers: {total_lines_written}")

if __name__ == "__main__":
    # For your case: folder = "lifestyle", output = "combined.txt", header separator matches your example
    concat_text_files_strip_header(
        folder_path="opinion",
        output_file="opinion_combined.txt",
        extension=".txt",
        header_separator="============================================================"
    )

Successfully processed 200 files. Combined output written to 'opinion_combined.txt'.
Total lines after stripping headers: 6032


In [5]:
import re

def clean_text(text):
    # 1. Fix apostrophe + capital
    text = re.sub(r"([’'])([A-Z])", r"\1 \2", text)
    
    # 2. Fix lowercase + uppercase (camelCase split)
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    
    # 3. Fix letter + digit / digit + letter
    text = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", text)
    text = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", text)
    
    # 4. Fix missing space after punctuation
    text = re.sub(r"([.!?,;:])([a-zA-Z0-9])", r"\1 \2", text)
    
    # 5. Fix missing space before punctuation (but keep apostrophes safe)
    text = re.sub(r"([a-zA-Z0-9])([.!?,;:])", r"\1 \2", text)
    
    # 6. Fix common merged words like "fromSeptember" → "from September"
    text = re.sub(r"(from|to|of|for|with|on|at|by|in|about)([A-Z])", r"\1 \2", text)
    
    # 7. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

# Test it
sample = """A new international airline is showing interest to fly to Bhairahawa’sGautam Buddha International Airport.

Hungarian airline Wizz Air is planning to fly to Nepal’s second international airport whichcame into operation on May 16from September.

Currently, onlyJazeera Airways has been operating its flightat the airport, but there are talks aboutFly Dubai and Air Arabia flying to the airportin the near future.

Wizz Air’s general sales agent in Nepal, Ravi Chandra Singh, says the airline is looking to fly its 190-seater aircraft to Nepal. The budget airlines will operate flights from Abu Dhabi, adds Singh.

“We hope religious tourism expands following the entry of another airline in the country’s new airport,” says Singh."""

print(clean_text(sample))

A new international airline is showing interest to fly to Bhairahawa’s Gautam Buddha International Airport . Hungarian airline Wizz Air is planning to fly to Nepal’s second international airport whichcame into operation on May 16 from September . Currently , only Jazeera Airways has been operating its flightat the airport , but there are talks about Fly Dubai and Air Arabia flying to the airportin the near future . Wizz Air’s general sales agent in Nepal , Ravi Chandra Singh , says the airline is looking to fly its 190-seater aircraft to Nepal . The budget airlines will operate flights from Abu Dhabi , adds Singh . “We hope religious tourism expands following the entry of another airline in the country’s new airport ,” says Singh .


In [11]:
import os
import re
from pathlib import Path

def clean_merged_words(text):
    """Clean merged words in text"""
    
    # 1. Fix apostrophe + capital (e.g., ’sGautam → ’s Gautam)
    text = re.sub(r"([’'])([A-Z])", r"\1 \2", text)
    
    # 2. Fix lowercase + uppercase (camelCase split)
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    
    # 3. Fix letter + digit / digit + letter
    text = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", text)
    text = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", text)
    
    # 4. Fix missing space after punctuation
    text = re.sub(r"([.!?,;:])([a-zA-Z0-9])", r"\1 \2", text)
    
    # 5. Fix missing space before punctuation (keep apostrophes safe)
    text = re.sub(r"([a-zA-Z0-9])([.!?,;:])", r"\1 \2", text)
    
    # 6. Fix common merged words with prepositions
    text = re.sub(r"(from|to|of|for|with|on|at|by|in|about)([A-Z])", r"\1 \2", text)
    
    # 7. Fix common merged words with "and"
    text = re.sub(r"(and)([A-Z])", r"\1 \2", text)
    
    # 8. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

def process_travel_folder(folder_path="English/economy"):
    """Process all txt files in the travel folder"""
    
    # Convert to Path object for easier handling
    folder = Path(folder_path)
    
    # Check if folder exists
    if not folder.exists():
        print(f"❌ Folder '{folder_path}' not found!")
        return
    
    # Find all .txt files
    txt_files = list(folder.glob("*.txt"))
    
    if not txt_files:
        print(f"⚠️  No .txt files found in '{folder_path}'")
        return
    
    print(f"📁 Found {len(txt_files)} text file(s) in '{folder_path}'\n")
    
    for file_path in txt_files:
        try:
            # Read the file
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Clean the content
            cleaned_content = clean_merged_words(content)
            
            # Check if any changes were made
            if content != cleaned_content:
                # Write back to the SAME file (overwrite)
                with open(file_path, 'w', encoding='utf-8') as file:
                    file.write(cleaned_content)
                print(f"✅ Cleaned: {file_path.name}")
            else:
                print(f"⏭️  No changes needed: {file_path.name}")
                
        except Exception as e:
            print(f"❌ Error processing {file_path.name}: {e}")

# Run the script
if __name__ == "__main__":
    process_travel_folder("English/economy")

📁 Found 400 text file(s) in 'English/economy'

✅ Cleaned: article_0116.txt
✅ Cleaned: article_0135.txt
✅ Cleaned: article_0045.txt
✅ Cleaned: article_0330.txt
✅ Cleaned: article_0155.txt
✅ Cleaned: article_0199.txt
✅ Cleaned: article_0086.txt
✅ Cleaned: article_0070.txt
✅ Cleaned: article_0068.txt
✅ Cleaned: article_0258.txt
✅ Cleaned: article_0059.txt
✅ Cleaned: article_0032.txt
✅ Cleaned: article_0141.txt
✅ Cleaned: article_0229.txt
✅ Cleaned: article_0175.txt
✅ Cleaned: article_0286.txt
✅ Cleaned: article_0119.txt
✅ Cleaned: article_0292.txt
✅ Cleaned: article_0041.txt
✅ Cleaned: article_0230.txt
✅ Cleaned: article_0060.txt
✅ Cleaned: article_0069.txt
✅ Cleaned: article_0180.txt
✅ Cleaned: article_0051.txt
✅ Cleaned: article_0093.txt
✅ Cleaned: article_0269.txt
✅ Cleaned: article_0109.txt
✅ Cleaned: article_0386.txt
✅ Cleaned: article_0145.txt
✅ Cleaned: article_0098.txt
✅ Cleaned: article_0245.txt
✅ Cleaned: article_0334.txt
✅ Cleaned: article_0344.txt
✅ Cleaned: article_0281.txt
✅

In [ ]:
import sys
import statistics
import time
from tiny_llm_scratch_with_tokenizer import PyNepBPETokenizer

VOCAB_TSV = 'nepbpe_vocab_bilingual.tsv'
#"dataset_ne/nepbpe_vocab_new.tsv"

# MUST be identical to what you trained with (train.py). If these differ,
# normalization drifts and surface lookups miss.
FOLDING_RULES = [
    ("सङ्ग", "संग"),
    ("सँग", "संग"),
]

# 'Ġ' (U+0120) is the byte-alphabet surface for space (0x20). Without
# Ġ-prefixing, each inter-word space is its own token.
SPACE_PIECE = "\u0120"


def show_piece(p: str) -> str:
    """Render a piece for display: space as ·, ZWNJ as <ZWNJ>."""
    if p == SPACE_PIECE:
        return "·"
    if p == "\u200c":
        return "<ZWNJ>"
    return p


SAMPLES = [
    "तिम्रो मुस्कानमा बिहानको उज्यालो भेटेँ",
    "तिम्रो आँखामा आफ्नै संसार देखेँ",
    "शब्दले भन्न नसक्ने भावना",
    "मुटुले चुपचाप तिमीलाई लेखेँ",
    "हावाले तिम्रो नाम बिस्तारै बोलाउँछ",
    "चन्द्रमाले तिम्रो यादमा रात सजाउँछ",
    "टाढा भए पनि मन नजिकै रहन्छ",
    "साँचो माया समयसँग कहिल्यै नहराउँछ",
    "तिमीसँग बितेको प्रत्येक पल",
    "जीवनको सबैभन्दा सुन्दर गीत बन्यो",
    "दुःखका बादल आए पनि",
    "तिम्रो साथले हरेक आँसु मुस्कान बन्यो",
    "माया भनेको केवल शब्द होइन",
    "एकअर्काको सपना बोक्ने यात्रा हो",
    "विश्वास, सम्मान र साथको डोरीले",
    "दुई आत्मालाई सधैं जोड्ने कथा हो",
    "यदि अर्को जन्मको कथा लेखियो भने",
    "फेरि पनि तिमी नै मेरो रोजाइ हुनेछौ",
    "यस जन्मझैं, त्यो जन्ममा पनि",
    "मेरो हरेक प्रार्थनाको उत्तर तिमी नै हुनेछौ",
   ' नेपाल (आधिकारिक नाम: सङ्घीय लोकतान्त्रिक गणतन्त्र नेपाल)'
]


def main(test_file=None) -> None:
    tok = PyNepBPETokenizer(folding_rules=FOLDING_RULES)
    n = tok.load_vocab_tsv(VOCAB_TSV)
    print(f"loaded {n} tokens from {VOCAB_TSV}\n")

    raw_rates, content_rates = [], []
    tok_total = word_total = space_total = fails = 0

    print("=== sample tokenization ===")
    for idx, s in enumerate(SAMPLES, 1):
        print(f"  [{idx}/{len(SAMPLES)}] processing...", end="", flush=True)

        ids = tok.encode(s)
        pieces = [tok.get_token_surface(i) for i in ids]
        norm = tok.normalize(s)
        words = max(1, len(norm.split()))
        spaces = sum(1 for p in pieces if p == SPACE_PIECE)
        content = len(ids) - spaces
        ok = tok.decode(ids) == norm

        raw_rates.append(len(ids) / words)
        content_rates.append(content / words)
        tok_total += len(ids)
        word_total += words
        space_total += spaces
        if not ok:
            fails += 1

        shown = " ".join(show_piece(p) for p in pieces)
        print(f"\r  {s}")
        print(
            f"    {len(ids)} tok = {content} content + {spaces} space | "
            f"{len(ids)/words:.2f}/word ({content/words:.2f} ex-space) | "
            f"roundtrip={'OK' if ok else 'FAIL'}"
        )
        print(f"    {shown}")
        if not ok:
            print(f"    DECODED : {tok.decode(ids)!r}")
            print(f"    EXPECTED: {norm!r}")

    print("\n=== sample summary ===")
    print(
        f"  tokens/word   : mean={statistics.mean(raw_rates):.2f}  "
        f"median={statistics.median(raw_rates):.2f}"
    )
    print(
        f"  ex-space/word : mean={statistics.mean(content_rates):.2f}  "
        f"median={statistics.median(content_rates):.2f}   <- real subword fertility"
    )
    print(
        f"  micro/word    : {tok_total/max(1,word_total):.2f}  "
        f"(space tokens = {space_total}/{tok_total} = "
        f"{100*space_total/max(1,tok_total):.0f}%)"
    )
    print(f"  roundtrip     : {len(SAMPLES)-fails}/{len(SAMPLES)} OK")

    # Optional: fertility over a held-out file (fast, uses the Rust encode path).
    if test_file:
        print(f"\n=== fertility over {test_file} ===")
        space_id = tok.vocab_get_id(SPACE_PIECE)  # int (or None), computed once
        tt = ww = ss = lines = 0
        t0 = time.perf_counter()
        try:
            with open(test_file, encoding="utf-8") as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue

                    if line_num % 1000 == 0:
                        elapsed = time.perf_counter() - t0
                        print(
                            f"  ... line {line_num}: {tt} tokens in {elapsed:.1f}s "
                            f"({tt/max(1,elapsed):.0f} tok/s)",
                            flush=True,
                        )

                    w = len(tok.normalize(line).split())
                    if w == 0:
                        continue

                    ids = tok.encode(line)
                    sp = ids.count(space_id) if space_id is not None else 0
                    tt += len(ids)
                    ss += sp
                    ww += w
                    lines += 1

                    if lines >= 20000:
                        print(f"  Reached {lines} lines limit", flush=True)
                        break

        except FileNotFoundError:
            print(f"  Error: File '{test_file}' not found. Skipping fertility analysis.")
            return
        except KeyboardInterrupt:
            print(f"\n  Interrupted after {lines} lines", flush=True)
            return

        dt = time.perf_counter() - t0
        print(
            f"  lines={lines} | tokens={tt} | tokens/word={tt/max(1,ww):.3f} | "
            f"ex-space/word={(tt-ss)/max(1,ww):.3f} | space-frac={ss/max(1,tt):.3f} | "
            f"{dt:.1f}s ({tt/max(1,dt):.0f} tok/s)"
        )


if __name__ == "__main__":
    # Handle both command-line and Jupyter environments.
    try:
        if len(sys.argv) > 1 and not sys.argv[1].startswith("--f="):
            main(sys.argv[1])
        else:
            main()
    except KeyboardInterrupt:
        print("\nInterrupted by user", file=sys.stderr)